# DKI on Google Colab

End-to-end run of the refactored DKI (Data-driven Keystone
Identification) framework. This notebook:

1. Clones the repo and installs deps.
2. Lets you use the bundled gLV data or upload your own abundance CSV
   (samples-as-rows **or** taxa-as-rows).
3. Trains the Phase-1 model (batched `dopri5` replicator ODE, cosine
   LR, early stopping, best-val checkpoint).
4. Predicts on the test set.
5. Computes **classical structural keystoneness** for every
   (sample, species) pair (Python port of `Keystoneness_computing.R`).
6. *(Optional)* Trains the **original DKI model** for head-to-head
   comparison.

Tip: switch the runtime to GPU (Runtime → Change runtime type → T4)
for a speedup on larger N — the trainer auto-detects CUDA / MPS / CPU.

## Research roadmap

This notebook currently runs **Phase 1**. The full refactor is staged
across six phases — each lands a self-contained improvement with its
own sanity check before the next is started.

| Phase | Status | What it adds |
|---|---|---|
| **1. Faithful refactor + batched integration** | ✅ landed | `dki/` package, batched `dopri5` replicator ODE (`rtol=1e-5`, `atol=1e-7`, t=[0, 100]), cosine LR, gradient clipping at 1.0, early stop on val BC, best-val checkpoint, auto CUDA/MPS/CPU. ~600× per-epoch speedup over the original. |
| **2. Nonlinear ODEFunc + composite loss** | planned | Per-capita fitness becomes `fc2(SiLU(fc1(y)))` with hidden dim 2N; the two-stacked-`Linear` construction in the original cNODE2 is provably equivalent to a single `Linear` (W2·W1 = W). Composite loss `α·BC + (1−α)·CLR-MSE` with α=0.3 to repair rare-species accuracy. Includes an expressivity regression test. |
| **3. Deep-equilibrium reformulation** | planned | `--mode deq`: solve the replicator fixed point directly with Anderson acceleration (50 iters, tol 1e-6), backprop via the implicit function theorem. Falls back to the ODE solver on non-convergence. Target ≥3× speedup with mean BC < 0.01 vs ODE. |
| **4. Ensembles + uncertainty + null-model normalisation** | planned | K=5 bootstrap-resampled models → predictions become `(mean, std)`. Keystoneness module gains an **alternative** z-score calibration (50 abundance-matched null species per (sample, species)) alongside — not replacing — the classical `(1−p)` formula. |
| **5. Shapley keystoneness** *(extension)* | planned | Monte-Carlo Shapley (N_perm=200) for a **different question** — synergy/redundancy-aware contribution — not a fix to the classical definition. Reported as `k_shapley_synergistic`, never replacing `k_classical` or `k_zscore`. |
| **6. Self-consistency regulariser** | planned | Training-time auxiliary loss: mask one present species, predict q', re-feed q'>0 through the model, require the second prediction matches q' under BC. Weighted by `λ_consistency=0.1`. Goal: reduce ensemble std of keystoneness predictions. |

Throughout: the original `DKI.py` is preserved at
`legacy/DKI_original.py`, `Keystoneness_computing.R` stays runnable as
a cross-check, and `pytest` covers simplex preservation, loss
correctness, batched-vs-loop equivalence, and the keystoneness port.

Design notes pinned by the project:
* The metacommunity assumption (same `f` across all samples, only `z`
  varies) is preserved — the ODEFunc takes **only `y`**; no covariate
  conditioning, hypernetworks, or context-dependent interactions.
* Classical Paine-style keystoneness stays in the default output; null
  z-score and Shapley land as alternatives, not replacements.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/peaceful-goodall-4AC8l'   # change to 'main' once merged
REPO_DIR = '/content/DKI'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
%cd $REPO_DIR
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from dki.device import auto_device
print('torch', torch.__version__, 'device', auto_device())

## 2. Pick your data

**Option A** — use the bundled gLV synthetic data (works out of the box).

**Option B** — upload your own CSV (numeric abundance table). The cell
below handles both orientations:
* `samples_as_rows=True`  → rows are samples, columns are taxa
* `samples_as_rows=False` → rows are taxa,   columns are samples (legacy)

Counts or relative abundances both work — each sample is renormalised to
sum to 1 internally.

In [ ]:
USE_BUNDLED = True              # set False to upload your own
samples_as_rows = True          # only matters when USE_BUNDLED=False
header_row = False              # set True if your CSV has a header row
index_col  = False              # set True if your CSV has a row-label column

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    import pandas as pd
    for name, _ in uploaded.items():
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if 'train' in name.lower() or len(uploaded)==1
                                       else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR)
!ls -la $DATA_DIR

## 3. Train the new model

In [ ]:
from dki.train import TrainConfig, train

cfg = TrainConfig(
    data_dir=DATA_DIR,
    out_dir='/content/results',
    epochs=1000,              # matches the original DKI.py budget; one
    batch_size=20,            # minibatch step per epoch -> ~50 passes
    lr=1e-2,                  # over a 400-sample train set.
    min_lr=1e-4,
    t_final=100.0,
    grad_clip=1.0,
    early_stop_patience=200,  # val BC is noisy; allow long plateaus.
    val_fraction=0.2,
    seed=0,
    save_predictions=True,
)
model, result, data = train(cfg)
print(f'\nBest val BC: {result.best_val_loss:.6f} at epoch {result.best_epoch}')
print(f'Mean epoch wall-clock: {np.mean(result.epoch_seconds):.3f}s')
print(f'Total wall-clock: {np.sum(result.epoch_seconds):.1f}s')

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(result.train_loss, label='train BC')
ax.plot(result.val_loss,   label='val BC')
ax.set_xlabel('epoch'); ax.set_ylabel('Bray-Curtis')
ax.set_title('DKI Phase-1 training')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Predict on test set

In [ ]:
from dki.infer import predict
from dki.losses import bray_curtis

if data.p_test is not None and data.z_test is not None:
    qtst = predict(model, data.z_test, t_final=cfg.t_final).cpu().numpy()
    qtrn = predict(model, data.z_all,  t_final=cfg.t_final).cpu().numpy()
    test_bc = bray_curtis(torch.from_numpy(qtst), data.p_test.cpu()).item()
    print(f'Test Bray-Curtis (mean over {qtst.shape[0]} samples): {test_bc:.4f}')
else:
    print('No test set found in', DATA_DIR)
    qtst = qtrn = None

## 6. Keystoneness

Computes the **classical structural keystoneness** from Wang et al.
(bioRxiv 2023.03.15.532858v3) for every (sample, species) pair listed
in `Sample_id.csv` / `Species_id.csv`:

$$
k_{\text{classical}}(i, s) \;=\; \mathrm{BC}\!\left(\tilde q_{i,\setminus s},\; q_{i,\setminus s}\right) \;\cdot\; (1 - p_{s,i})
$$

where $\tilde q_{i,\setminus s}$ is the full-community prediction with
species $s$ masked-and-renormalised, and $q_{i,\setminus s}$ is the
model's leave-one-out prediction.

Phase 4 (planned) will add an alternative null-model z-score calibration;
Phase 5 (planned) will add a synergy-aware Shapley extension. Both will
be reported alongside `k_classical`, not replacing it.

In [ ]:
import pandas as pd
from dki.keystoneness import classical_structural_keystoneness

sample_id  = np.loadtxt(os.path.join(DATA_DIR, 'Sample_id.csv'),  delimiter=',').astype(int)
species_id = np.loadtxt(os.path.join(DATA_DIR, 'Species_id.csv'), delimiter=',').astype(int)

# Legacy on-disk orientation: (n_species, n_samples_or_pairs).
Ptrain = np.loadtxt(os.path.join(DATA_DIR, 'Ptrain.csv'), delimiter=',')
Ptest  = np.loadtxt(os.path.join(DATA_DIR, 'Ptest.csv'),  delimiter=',')
Ptrain = Ptrain / Ptrain.sum(axis=0, keepdims=True).clip(min=1e-12)
Ptest  = Ptest  / Ptest.sum(axis=0, keepdims=True).clip(min=1e-12)

kdf = classical_structural_keystoneness(
    qtrn=qtrn, qtst=qtst, ptrn=Ptrain, ptst=Ptest,
    sample_id=sample_id, species_id=species_id,
)
kdf.head()

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(kdf['k_true'], kdf['k_pred'])
print(f'Spearman(k_true, k_pred) = {rho:.3f}  (n={len(kdf)}, p={pval:.2g})')

fig, ax = plt.subplots(figsize=(5,5))
ax.hexbin(kdf['k_true'], kdf['k_pred'], gridsize=40, mincnt=1, cmap='Spectral_r')
lim = max(kdf['k_true'].max(), kdf['k_pred'].max())
ax.plot([0, lim], [0, lim], color='#d01c8b', lw=1)
ax.set_xlabel(r'$k_\mathrm{classical}$ (true)')
ax.set_ylabel(r'$k_\mathrm{classical}$ (predicted)')
ax.set_title(f'Structural keystoneness   Spearman $\\rho$={rho:.2f}')
plt.show()

kdf.to_csv('/content/results/keystoneness.csv', index=False)
print('Saved /content/results/keystoneness.csv')

In [ ]:
# Top-10 predicted keystone species (highest k_pred)
kdf.sort_values('k_pred', ascending=False).head(10)

## 7. (Optional) Compare to the original DKI model

Trains the legacy 2-Linear ODEFunc on the same split, same seed, using
the unchanged training loop (per-sample for-loop, 10,000-step Euler
integration). On a Colab CPU this is **~170 s / epoch** for the gLV
data, so the default below is 3 epochs (~9 minutes).

The new model on the same 3-5 epochs already matches or beats the
legacy val BC, and is ~600× faster per epoch.

Set `RUN_LEGACY = True` to enable.

In [ ]:
RUN_LEGACY = False
LEGACY_EPOCHS = 3

if RUN_LEGACY:
    out = subprocess.run([
        sys.executable, 'legacy/baseline_runner.py',
        '--data', DATA_DIR,
        '--epochs', str(LEGACY_EPOCHS),
        '--seed', str(cfg.seed),
        '--val-fraction', str(cfg.val_fraction),
        '--out', '/content/results/legacy',
    ], capture_output=True, text=True)
    print(out.stdout)
    if out.returncode != 0:
        print('STDERR:', out.stderr)
else:
    print('Skipping legacy run. Flip RUN_LEGACY = True to enable.')

In [ ]:
# Plot val BC: new (200 ep) vs legacy (LEGACY_EPOCHS ep).
if RUN_LEGACY and os.path.exists('/content/results/legacy/val_loss.npy'):
    legacy_val = np.load('/content/results/legacy/val_loss.npy')
    legacy_t   = np.load('/content/results/legacy/epoch_times.npy')
    new_val    = np.array(result.val_loss)
    new_t      = np.array(result.epoch_seconds)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(new_val,    label=f'new  (mean {new_t.mean():.2f}s/ep)')
    axes[0].plot(legacy_val, label=f'legacy (mean {legacy_t.mean():.1f}s/ep)', marker='o')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val BC')
    axes[0].set_title('Val Bray-Curtis vs epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(np.cumsum(new_t),    new_val,    label='new')
    axes[1].plot(np.cumsum(legacy_t), legacy_val, label='legacy', marker='o')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('cumulative wall-clock (s, log)')
    axes[1].set_ylabel('val BC')
    axes[1].set_title('Val Bray-Curtis vs compute'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    speedup = legacy_t.mean() / new_t.mean()
    print(f'Per-epoch speedup: {speedup:.0f}x  '
          f'(legacy {legacy_t.mean():.1f}s vs new {new_t.mean():.2f}s)')
    print(f'Best val BC — new: {min(new_val):.4f}    legacy: {min(legacy_val):.4f}')
else:
    print('Legacy comparison disabled or not yet run.')

## 8. Save and download artifacts

In [ ]:
!ls -la /content/results
# Uncomment to download:
# from google.colab import files
# files.download('/content/results/best_model.pt')
# files.download('/content/results/qtst.csv')
# files.download('/content/results/qtrn.csv')
# files.download('/content/results/keystoneness.csv')

---

**What's next (work-in-progress):**

* Phase 2 — nonlinear `fc2(SiLU(fc1(y)))` per-capita fitness + composite
  Bray-Curtis + CLR loss.
* Phase 3 — Deep-equilibrium fixed-point solver (`--mode deq`).
* Phase 4 — K=5 ensemble + null-model z-score keystoneness (alongside
  classical, not replacing it).
* Phase 5 — Monte-Carlo Shapley keystoneness (synergy-aware extension).
* Phase 6 — Self-consistency regulariser.